In [ ]:
"""Correcting cash balances with manual journal entries

Demonstration of how to model manual journal entries in LUSID

Attributes
----------
reconciliations
cocoon
holdings
transaction configuration
cancel transactions
"""

## Correcting cash balances with manual journal entries

In this notebook, we demonstrate how users can create manual journal entries in LUSID. For the purposes of this notebook, we will consider the scenario where a portfolio's custodian has included a stock exchange fee of £5000 in its GBP cash balance calculation. The same fee has not been included in the IBOR. This might lead a portfolio manager to go into overdraft if they trade on that amount. Therefore we create a manual entry in LUSID while the reconcilations team investigate the break.

### Setup LUSID

In [ ]:
# Import LUSID
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
import finbourne_sdk_utils.cocoon.cocoon as cocoon
from finbourne_sdk_utils.cocoon.utilities import create_scope_id
from finbourne_sdk_utils.cocoon.seed_sample_data import seed_data
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.cocoon.cocoon_printer import format_transactions_response

# Import Libraries
import pprint
import pytz
import pandas as pd
import numpy as np
import json
import requests
import os
import warnings
from datetime import datetime, timedelta, time

# Configure notebook logging and warnings
import logging

logger = logging.getLogger()
logger.setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook",
)

print("LUSID Environment Initialised")
print(
    "LUSID API Version: ",
    api_factory.build(lu.ApplicationMetadataApi)
    .get_lusid_versions()
    .build_version,
)

### 1) Prepare setup data

In this notebook we have a portfolio called GLOBAL-EQUITY. 

In [ ]:
# Portfolio code
portfolio_code = "GLOBAL-EQUITY"

# Load a mapping file for loading data
with open(r"config/seed_data.json") as mappings_file:
    seed_data_mapping = json.load(mappings_file)

# Load a file to format holding response
with open(r"config/format_holdings_response.json") as mappings_file:
    format_holdings_response = json.load(mappings_file)

# Load transaction file
transactions_file = r"data/manual_cash_data.csv"
transactions_df = pd.read_csv(transactions_file)

# Define transaction portfolio API
transaction_portfolios_api = api_factory.build(lu.TransactionPortfoliosApi)

In [ ]:
# Define function to load holdings DF


def get_holdings_df(scope, code, date=datetime.now(pytz.UTC)):

    holdings_response = transaction_portfolios_api.get_holdings(
        scope=scope, code=code, property_keys=["Instrument/default/Name"]
    )

    holdings_df = lusid_response_to_data_frame(
        holdings_response, rename_properties=True
    )

    return holdings_df

### 2) Load IBOR data

We have transactions from the IBOR which we load into the <b>ibor-nb</b> `scope`.

In [ ]:
# The seed_data() function takes a file of transaction data
# and loads portfolios, instruments, and transactions into LUSID
# We use this function as a quick way of generating a demo portfolio

ibor_df = transactions_df[transactions_df["scope"] == "IBOR"]
ibor_df.drop(columns=["scope"], inplace=True)

ibor_scope = "ibor-nb123"

seed_data_response = seed_data(
    api_factory,
    ["portfolios", "instruments", "transactions"],
    ibor_scope,
    ibor_df,
    "DataFrame",
    mappings=seed_data_mapping,
)

print(
    f"Portfolio {portfolio_code} has been created in scope {ibor_scope} with transactions."
)

Show IBOR holdings:

* The IBOR has 500,000 GBP in Cash

In [ ]:
get_holdings_df(ibor_scope, "GLOBAL-EQUITY")

### 3) Load Custodian data

We have transactions from the IBOR which we load into the <b>custodian-nb</b> `scope`.

In [ ]:
# The seed_data() function takes a file of transaction data
# and loads portfolios, instruments, and transactions into LUSID
# We use this function as a quick way of generating a demo portfolio

custodian_df = transactions_df[transactions_df["scope"] == "Custodian"]
custodian_df.drop(columns=["scope"], inplace=True)

custodian_scope = "custodian-nb123"

seed_data_response = seed_data(
    api_factory,
    ["portfolios", "instruments", "transactions"],
    custodian_scope,
    custodian_df,
    "DataFrame",
    mappings=seed_data_mapping,
)

print(
    f"Portfolio {portfolio_code} has been created in scope {custodian_scope} with transactions."
)

Show Custodian holdings:

* The Custodian has 595,000 GBP in Cash

In [ ]:
get_holdings_df(custodian_scope, "GLOBAL-EQUITY")

### 4) Reconcile Custodian versus IBOR

We use LUSID's holdings [reconcilation functionality](https://support.finbourne.com/how-do-i-reconcile-my-holdings-in-lusid) to reconcile the IBOR's view against the Custodian's view of GLOBAL-EQUITY. 

In [ ]:
def run_ibor_cust_recon(statement_datetime):

    ibor_portfolio = models.PortfolioReconciliationRequest(
        portfolio_id=models.ResourceId(scope=ibor_scope, code=portfolio_code),
        effective_at=statement_datetime,
        as_at=statement_datetime,
    )

    # Define our fund accountant portfolio
    custodian_portfolio = models.PortfolioReconciliationRequest(
        portfolio_id=models.ResourceId(scope=custodian_scope, code=portfolio_code),
        effective_at=statement_datetime,
        as_at=statement_datetime,
    )

    # Create our reconciliation request
    reconcile_holdings_request = models.PortfoliosReconciliationRequest(
        left=ibor_portfolio,
        right=custodian_portfolio,
        instrument_property_keys=["Instrument/default/Name"],
    )

    # Reconcile the two portfolios
    reconciliation = api_factory.build(lu.ReconciliationsApi).reconcile_holdings(
        portfolios_reconciliation_request=reconcile_holdings_request
    )

    return reconciliation

In [ ]:
first_recon_datetime = datetime.now(pytz.UTC).isoformat()
recon_result = run_ibor_cust_recon(first_recon_datetime)

print(f"The AsAt time for the recon is: {first_recon_datetime}")

### 5) Result: we have a break of £5000!

In [ ]:
lusid_response_to_data_frame(recon_result, rename_properties=True)

### 6) Create a manual journal entry to correct the break

In [ ]:
transactions_file = r"data/break_correction.csv"
break_correction_df = pd.read_csv(transactions_file)
break_correction_df["portfolio_code"] = portfolio_code
break_correction_df["txn_type"] = "ManualEntryCashOut"
break_correction_df.drop(columns=["scope"], inplace=True)
break_correction_df

In [ ]:
transaction_mapping = {
    "identifier_mapping": {
        "ClientInternal": "instrument_id",
        "Currency": "cash_transactions",
    },
    "required": {
        "code": "portfolio_code",
        "transaction_id": "txn_id",
        "type": "txn_type",
        "transaction_price.price": "txn_price",
        "transaction_price.type": "$Price",
        "total_consideration.amount": "txn_consideration",
        "units": "txn_units",
        "transaction_date": "txn_trade_date",
        "total_consideration.currency": "currency",
        "settlement_date": "txn_settle_date",
    },
    "optional": {},
    "properties": [],
}

In [ ]:
result = cocoon.load_from_data_frame(
    api_factory=api_factory,
    scope=ibor_scope,
    data_frame=break_correction_df,
    mapping_required=transaction_mapping["required"],
    mapping_optional=transaction_mapping["optional"],
    file_type="transactions",
    identifier_mapping=transaction_mapping["identifier_mapping"],
    property_columns=transaction_mapping["properties"],
    properties_scope=ibor_scope,
)

succ, failed = format_transactions_response(result)
print(f"number of successful portfolios requests: {len(succ)}")
print(f"number of failed portfolios requests    : {len(failed)}")

### 7) Create a new transaction type for the journal entry

In [ ]:
movement = models.TransactionTypeMovement(
    movement_types="CashAccrual",
    side="Side1",
    direction=-1,
    properties={},
    mappings=[],
)

alias = models.TransactionTypeAlias(
    type="ManualEntryCashOut",
    description="Booking of manual cash out ledgder entry",
    transaction_class="JournalEntry",
    transaction_roles="Shorter",
)
response = api_factory.build(lu.TransactionConfigurationApi).set_transaction_type(
    source="default",
    type="ManualEntryCashOut",
    transaction_type_request=models.TransactionTypeRequest(
        aliases=[alias],
        movements=[movement],
    )
)

### 8) Rerun the reconcilation

The result is empty - there are no breaks.

In [ ]:
second_recon_datetime = datetime.now(pytz.UTC).isoformat()
second_recon_response = run_ibor_cust_recon(second_recon_datetime).values

print(second_recon_response)
print(f"The AsAt time for the recon is: {second_recon_datetime}")

### Cleanup

Cancel the adjustment to keep notebook idempotent.

In [ ]:
cancel_response = transaction_portfolios_api.cancel_transactions(
    scope=ibor_scope, code=portfolio_code, transaction_ids=["cash_003"]
)

cancel_time = cancel_response.as_at

print(f"The  cancel datetime request is: {cancel_time}")